# Zeek Log Cleaning (zat-based)

This notebook reads common Zeek log types using **zat**, applies light normalization + stable pseudonymization (reusing `mappings/`), and writes cleaned outputs to `data/cleaned_zeek/`.

Supported log types:
- `conn.log`
- `dns.log`
- `http.log`
- `ssl.log`
- `files.log`
- `ssh.log`


In [25]:
# --- 0) Install & import dependencies (zat) ---
import sys, subprocess
import os, json, hashlib, re
from pathlib import Path
import pandas as pd
from zat.log_to_dataframe import LogToDataFrame
from collections import defaultdict

pd.set_option("display.max_columns", 200)


In [26]:
# --- 1) Paths & log list ---
PROJECT_ROOT = Path.cwd().parent.resolve()

DATA_DIR = PROJECT_ROOT / "data"         # put *.log here
OUT_DIR = PROJECT_ROOT / "sanidata"
MAP_DIR = PROJECT_ROOT / "mappings"    # reuse existing mapping dir

OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1) Zeek log "types" we care about (filename-based) ---
ZEEK_FILES = {
    "conn":  "conn.log",
    "dns":   "dns.log",
    "http":  "http.log",
    "ssl":   "ssl.log",
    "files": "files.log",
    "ssh":   "ssh.log",
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][log_type] -> list[Path]  (1 or many)
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for log_type, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][log_type].append(p)
    return inputs

INPUTS = collect_inputs(DATA_DIR, ZEEK_FILES)

# Quick sanity print: how many matches per scenario/log_type
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")


print("Input dir:", DATA_DIR)
print("Output dir:", OUT_DIR)
print("Mappings dir:", MAP_DIR)


sc3: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
sc4: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
sc7: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
Input dir: /Users/zhuoran/Projects/SSCMDataset/data
Output dir: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mappings dir: /Users/zhuoran/Projects/SSCMDataset/mappings


In [27]:
# --- 2) Stable SALT (reuse same pattern) ---
SALT_PATH = MAP_DIR / "salt.txt"
if SALT_PATH.exists():
    SALT = SALT_PATH.read_text().strip()
else:
    SALT = hashlib.sha256(os.urandom(32)).hexdigest()
    SALT_PATH.write_text(SALT)


In [28]:

def _load_map(name: str) -> dict:
    p = MAP_DIR / f"{name}.json"
    if p.exists():
        return json.loads(p.read_text())
    return {}

def _save_map(name: str, m: dict):
    p = MAP_DIR / f"{name}.json"
    p.write_text(json.dumps(m, ensure_ascii=False, indent=2))

def stable_hash(text: str) -> str:
    return hashlib.sha256((SALT + str(text)).encode("utf-8")).hexdigest()

def map_token(value, prefix: str, map_name: str, digits: int = 6):
    """Map a raw value to a stable pseudonym like ip_012345, user_000123, etc.
    Uses mappings/<map_name>.json for persistence across runs.
    """
    m = _load_map(map_name)
    key = str(value)
    if key in m:
        return m[key]

    h = stable_hash(key)
    mod = 10 ** digits
    n = int(h[:16], 16) % mod
    token = f'{prefix}_{n:0{digits}d}'

    # Resolve rare collisions: if token already used for a different key, walk forward
    used = set(m.values())
    while token in used:
        n = (n + 1) % mod
        token = f'{prefix}_{n:0{digits}d}'

    m[key] = token
    _save_map(map_name, m)
    return token

def normalize_scalar(x):
    """Normalize common Zeek placeholders."""
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s in {"-", ""}:
            return None
        if s.lower() in {"(empty)", "null", "none", "nan"}:
            return None
        return s
    return x


In [ ]:
# --- 3) Sanitization policy (newt-only user; Azure-only FQDN) ---
# Only pseudonymize this username (case-insensitive). Everything else is preserved.
ONLY_PSEUDONYMIZE_USERS = {'newt'}

# Azure/Azure-hosted suffix allowlist (acts like *.suffix wildcard)
AZURE_FQDN_SUFFIXES = (
    'trafficmanager.net',
    'azure.com',
    'azure.net',
    'azureedge.net',
    'cloudapp.azure.com',
    'azurewebsites.net',
    'windows.net',
    'core.windows.net',
    'database.windows.net',
    'redis.cache.windows.net',
    'servicebus.windows.net',
    'vault.azure.net',
    'azure-api.net',
)

def _is_empty(v) -> bool:
    return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and v.strip() == '')

def is_azure_fqdn(fqdn: str) -> bool:
    if _is_empty(fqdn):
        return False
    f = str(fqdn).strip().lower().rstrip('.')
    return any(f == sfx or f.endswith('.' + sfx) for sfx in AZURE_FQDN_SUFFIXES)

def sanitize_fqdn(fqdn: str) -> str:
    """Pseudonymize Azure/Azure-hosted FQDNs only; keep all other FQDNs unchanged."""
    if _is_empty(fqdn) or pd.isna(fqdn):
        return fqdn
    s = str(fqdn).strip()
    base = s.rstrip('.')
    if not is_azure_fqdn(base):
        return s
    tok = map_token(base.lower(), 'FQDN', 'fqdn_map', digits=4)
    # keep a plausible domain shape without leaking the original
    return f'{tok}.example'

def sanitize_user_token(user: str) -> str:
    """Pseudonymize only allowlisted user tokens (case-insensitive)."""
    if _is_empty(user) or pd.isna(user):
        return user
    u = str(user).strip()
    # DOMAIN\\USER -> only consider USER for allowlist
    if '\\' in u:
        dom, name = u.rsplit('\\', 1)
        name_norm = name.strip().lower()
        if name_norm in ONLY_PSEUDONYMIZE_USERS:
            return dom + '\\' + map_token(name_norm, 'USER', 'user_map', digits=3)
        return u
    name_norm = u.lower()
    if name_norm in ONLY_PSEUDONYMIZE_USERS:
        return map_token(name_norm, 'USER', 'user_map', digits=3)
    return u



In [29]:
# --- 3) ZAT reader ---
log_reader = LogToDataFrame()

def read_zeek_log(path: Path) -> pd.DataFrame:
    df = log_reader.create_dataframe(str(path))
    # Standardize column names (zat already uses the #fields names)
    df.columns = [c.strip() for c in df.columns]
    return df


In [30]:
USER_COL_CANDIDATES = {
    "user", "username", "auth_user", "client_user", "server_user", "uid_user"
}

FQDN_COL_PATTERNS = [
    re.compile(r'(^|\.)query$'),
    re.compile(r'(^|\.)host$'),
    re.compile(r'(^|\.)server_name$'),
    re.compile(r'(^|\.)sni$'),
    re.compile(r'(^|\.)domain$'),
]

def pick_user_cols(cols):
    out = []
    for c in cols:
        if c.lower() in USER_COL_CANDIDATES:
            out.append(c)
    return out


def pick_fqdn_cols(cols):
    out = []
    for c in cols:
        for p in FQDN_COL_PATTERNS:
            if p.search(c.lower()):
                out.append(c)
                break
    return out

def clean_zeek_df(df: pd.DataFrame, log_type: str) -> pd.DataFrame:
    # Normalize placeholders
    for c in df.columns:
        df[c] = df[c].map(normalize_scalar)

    # User columns: newt-only
    for c in pick_user_cols(df.columns):
        df[c] = df[c].map(sanitize_user_token)

    # FQDN columns: Azure-only
    for c in pick_fqdn_cols(df.columns):
        df[c] = df[c].map(sanitize_fqdn)

    # Helpful standard cols
    df.insert(0, 'log_type', log_type)
    if 'ts' in df.columns:
        df['ts_utc'] = pd.to_datetime(df['ts'], unit='s', utc=True, errors='coerce')
    return df
 


In [31]:
# --- 5) Process all logs ---
results = {}

for sc, by_type in INPUTS.items():
    sc_out = OUT_DIR / sc
    sc_out.mkdir(parents=True, exist_ok=True)

    for log_type, paths in by_type.items():
        if not paths:
            continue

        # If multiple files per type, concatenate
        frames = []
        for p in sorted(paths):
            df = read_zeek_log(p)
            frames.append(df)
        df_all = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

        df_clean = clean_zeek_df(df_all, log_type)
        results[(sc, log_type)] = df_clean

        out_csv  = sc_out / f'{log_type}_cleaned.csv'
        out_parq = sc_out / f'{log_type}_cleaned.parquet'
        df_clean.to_csv(out_csv, index=False)
        df_clean.to_parquet(out_parq, engine='fastparquet', index=False)
        print(f'[{sc}] {log_type}: wrote {out_csv.name}, {out_parq.name}')

print('\nDone. Cleaned:', len(results), 'scenario/log-type tables')


Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/conn_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/conn_cleaned.parquet
[sc7] conn: /Users/zhuoran/Projects/SSCMDataset/data/sc7/conn.log -> conn_cleaned.csv, conn_cleaned.parquet
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/dns_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/dns_cleaned.parquet
[sc7] dns: /Users/zhuoran/Projects/SSCMDataset/data/sc7/dns.log -> dns_cleaned.csv, dns_cleaned.parquet
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/http_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/http_cleaned.parquet
[sc7] http: /Users/zhuoran/Projects/SSCMDataset/data/sc7/http.log -> http_cleaned.csv, http_cleaned.parquet
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/ssl_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/ssl_cleaned.parquet
[sc7] ssl: /Users/zhuoran/Projects/SSCMDataset/data/sc7/ssl.log -> ssl_cleaned.csv, 

In [32]:
# --- 6) Quick peek ---
for lt, df in results.items():
    print("\n===", lt, "===")
    display(df.head(3))
